# 01. Preprocessing pipeline

Builds the preprocessing pipeline and fits it on the training split (`split_3 == "train"`) only.
The fitted pipeline is saved to `outputs/preprocessor.joblib` and loaded by notebooks 02-04.
Variable coding is described in Supplementary Table 1 of the manuscript.

| Group | Columns | Processing |
|---|---|---|
| Numeric | Age, BMI, smoking amount/duration/quit days, FVC, FEV1, DLCO | median imputation, min-max scaling |
| Biomarkers | VEGF, EGFR, Ki-67, p53 | -1 (not performed) treated as missing, mode imputation |
| Symptoms | symptom_1 ... symptom_12 | binary indicators |
| Sex | Gender | one-hot (first level dropped) |
| Categorical | T/N/M stage, histologic subtype, resection status, smoking history, operative extent, adjuvant therapy | one-hot |

Columns not listed above are dropped by the pipeline (`remainder="drop"`).

In [ ]:
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sksurv.util import Surv

warnings.filterwarnings("ignore", category=FutureWarning)

DATA_PATH    = Path("../data/dummy_data.csv")     # replace with your own data
OUTPUT_DIR   = Path("../outputs"); OUTPUT_DIR.mkdir(exist_ok=True)
PREPROC_PATH = OUTPUT_DIR / "preprocessor.joblib"  # written by 01_data_preprocessor
MODEL_PATH   = OUTPUT_DIR / "RSF_model.joblib"     # written by 02_RSF_train
RANDOM_SEED  = 42

TIME_COL, EVENT_COL, ID_COLS = "OS", "event", ["SubjectKey"]   # OS in days

def load_splits(path=DATA_PATH):
    """Return {"Train"|"Val"|"Test": (X_raw, y)} using the column split_3."""
    df = pd.read_csv(path)
    df[EVENT_COL] = df[EVENT_COL].astype(bool)
    out = {}
    for name, key in [("Train", "train"), ("Val", "val"), ("Test", "test")]:
        d = df[df["split_3"] == key]
        out[name] = (d.drop(columns=ID_COLS + [TIME_COL, EVENT_COL]),
                     Surv.from_arrays(d[EVENT_COL], d[TIME_COL]))
    return out

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, MinMaxScaler, OneHotEncoder

NUM_COLS        = ["Age", "BMI", "Smoke Amount", "Smoke Duration", "Smoke Quit_days",
                   "Spirometry FVC", "Spirometry FEV1", "Diffusion DLCO"]
BIOMARKER_COLS  = ["VEGF", "EGFR", "Ki-67", "p53"]
SYMPTOM_COLS    = [f"symptom_{i}" for i in range(1, 13)]
SIMPLE_CAT_COLS = ["Gender"]
ONEHOT_COLS     = ["T_stage", "N_stage", "M_stage", "Adeno type", "resection type",
                   "Smoke Hx", "Opextent", "Postop Adjuvant Therapy"]

preprocessor = ColumnTransformer(remainder="drop", transformers=[
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale",  MinMaxScaler())]), NUM_COLS),
    ("biomarker", SimpleImputer(missing_values=-1, strategy="most_frequent"), BIOMARKER_COLS),
    ("symptom", FunctionTransformer(np.isin, kw_args={"test_elements": [1]}), SYMPTOM_COLS),
    ("simplecat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                            ("ohe",    OneHotEncoder(drop="first"))]), SIMPLE_CAT_COLS),
    ("onehot", OneHotEncoder(handle_unknown="ignore"), ONEHOT_COLS),
])

In [ ]:
splits = load_splits()
for s, (X_raw, y) in splits.items():
    print(f"{s:5s}: n = {len(y):4d}, deaths = {int(y['event'].sum())}")

X_train = preprocessor.fit_transform(splits["Train"][0])   # fitted on the training set only
X_val   = preprocessor.transform(splits["Val"][0])
X_test  = preprocessor.transform(splits["Test"][0])
print(f"Encoded shapes -> train {X_train.shape}, val {X_val.shape}, test {X_test.shape}")

joblib.dump(preprocessor, PREPROC_PATH)
print("Saved:", PREPROC_PATH)